# Comparable-company selection: reviewer labels and the fixed holdout

This report extends Michael's 27 October SIC2 v0 baseline with three independent reviewer label sets and the **160-company holdout**. It uses the same 373-company training pool and the baseline's candidate, peer-selection, and valuation functions. Company identity remains one `IQ_COMPANY_ID`; source-checked business activities are stored under that identifier.

The 36 reviewed pairs cover two development targets (PCTY and WEN). They assess **peer relevance**, not facet-fill accuracy. The holdout evaluates EV/EBITDA estimation; it does not validate private-company prediction, because its targets and peers are public companies.

In [1]:
from pathlib import Path
import json
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "v0/README.md").exists() and (p / "src/project_data.py").exists())
sys.path[:0] = [str(ROOT / "v0"), str(ROOT / "src")]
OUT = ROOT / "outputs-local/comps_pilot_targets"
GOLD = OUT / "golden_set/reviewer_evaluation"
FACETS = OUT / "facet_eval/20261028T135602Z"
SPLIT = OUT / "business_structure_refinement"
HOLDOUT = OUT / "holdout_completion"
pd.set_option("display.max_colwidth", 55)

## 1. Fixed inputs and comparison rules

The frozen v0 split is stratified on SIC2, after the team's 3–30 EV/EBITDA eligibility rule reduces the 560-company list to 533 public firms. The financial and SIC fields come from the saved `company_list_v3` Capital IQ snapshot; this run does not query the Capital IQ API. All holdout targets are compared only with training firms. Once eligibility is fixed, peer-selection calls do not read financial multiples; the valuation stage reads them after peer lists are produced. The embedding cache is checked against the team's saved candidate scores. The LLM routes read company names, websites, and business facets, without multiples or reviewer labels. The holdout LLM backend is recorded separately from the original Haiku training run.

In [2]:
candidate_run = json.loads((SPLIT / "holdout_all_candidates/run.json").read_text())
split = pd.read_csv(OUT / "split" / candidate_run["split_run"] / "split.csv", dtype=str)
print(split["sample"].value_counts().to_string())
print(f"Holdout candidates: {candidate_run['pairs']:,}; "
      f"maximum saved-score discrepancy: {candidate_run['max_saved_pair_cosine_difference']:.2g}")

sample
training    373
holdout     160
Holdout candidates: 2,195; maximum saved-score discrepancy: 7.4e-07


## 2. Peer labels and facet selection

Riya, Barbara, and Matthew independently scored the same 36 pairs from 0 to 2. A majority label is used only when at least two reviewers chose the same score; four three-way ties remain unresolved. Fiona's facet evaluator ranks peers from saved per-facet cosine scores. Each ablation removes one facet without fitting weights. The comparison below reports pairwise ranking accuracy on those labeled pairs; these two targets are development evidence, not an independent holdout.

Route precision and NDCG use only the reviewed candidates in the 36-pair pool. A shorter selected list can therefore have high precision with low coverage; the route table reports its length.

In [3]:
gold_run = json.loads((GOLD / "run.json").read_text())
facet_scores = pd.read_csv(FACETS / "summary.csv")
key = facet_scores[facet_scores["variant"].isin(
    ["generic (step 4)", "generic minus F3 primary_services"])]
show = key[["variant", "labels", "labeled pairs", "pairwise accuracy",
            "recall@10 of best comps", "top-10 labeled share"]].copy()
display(show.round(3))
print("Reviewer agreement:")
display(pd.read_csv(GOLD / "reviewer_agreement.csv").round(3))
routes_gold = pd.read_csv(GOLD / "route_metrics.csv")
display(routes_gold[routes_gold["target"].eq("mean")][[
    "reviewer", "route", "listed in top 10", "precision@10", "direct hits@10",
    "NDCG@10"]].round(3))
route_ndcg = routes_gold[routes_gold["target"].eq("mean")].pivot(
    index="reviewer", columns="route", values="NDCG@10")
fig_route, ax_route = plt.subplots(figsize=(9, 3.5))
route_ndcg.plot.bar(ax=ax_route)
ax_route.set_ylabel("NDCG@10 on reviewed peers")
ax_route.set_xlabel("Reviewer")
ax_route.set_ylim(0, 1)
ax_route.legend(title="Route", fontsize=7, loc="upper left", bbox_to_anchor=(1, 1))
fig_route.tight_layout()
fig_route.savefig(GOLD / "route_ndcg_by_reviewer.png", dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(9.5, 3.5))
chart = key.pivot(index="labels", columns="variant", values="pairwise accuracy")
chart.index = chart.index.str.replace("gold:facet_labels_", "", regex=False)
chart.plot.bar(ax=ax, color=["#2f6b91", "#e09b47"])
ax.set_ylabel("Pairwise ranking accuracy")
ax.set_xlabel("Reviewer labels")
ax.set_ylim(0, 1)
ax.legend(title="Facet set", fontsize=8, loc="upper left", bbox_to_anchor=(1, 1))
fig.tight_layout()
fig.savefig(FACETS / "reviewer_f3_pairwise.png", dpi=180)
plt.show()

,variant,labels,labeled pairs,pairwise accuracy,recall@10 of best comps,top-10 labeled share
0,generic (step 4),gold:facet_labels_riya,36,0.825,0.641,1.00
1,generic (step 4),gold:facet_labels_barbara,36,0.790,1.000,1.00
2,generic (step 4),gold:facet_labels_matthew,36,0.846,1.000,1.00
3,generic (step 4),gold:facet_labels_majority,32,0.851,1.000,0.95
12,generic minus F3 primary_services,gold:facet_labels_riya,36,0.925,0.885,0.95
13,generic minus F3 primary_services,gold:facet_labels_barbara,36,0.888,1.000,0.95
14,generic minus F3 primary_services,gold:facet_labels_matthew,36,0.874,1.000,0.95
15,generic minus F3 primary_services,gold:facet_labels_majority,32,0.921,1.000,0.80


Reviewer agreement:


,reviewer_a,reviewer_b,exact_agreement,pairs,agreement_rate
0,riya,barbara,12,36,0.333
1,riya,matthew,12,36,0.333
2,barbara,matthew,30,36,0.833


,reviewer,route,listed in top 10,precision@10,direct hits@10,NDCG@10
8,riya,cosine similarity,10.0,0.95,5.0,0.871
9,riya,cosine then LLM good comps,4.0,1.00,3.5,0.591
10,riya,LLM picks directly,4.0,1.00,3.5,0.569
11,riya,LLM picks then cosine,4.0,1.00,3.5,0.569
20,barbara,cosine similarity,10.0,0.50,1.5,0.844
21,barbara,cosine then LLM good comps,4.0,1.00,1.5,0.852
22,barbara,LLM picks directly,4.0,1.00,1.5,0.796
23,barbara,LLM picks then cosine,4.0,1.00,1.5,0.796
32,matthew,cosine similarity,10.0,0.45,2.5,0.910
33,matthew,cosine then LLM good comps,4.0,1.00,2.5,0.907


C:\Users\13156\AppData\Local\Temp\ipykernel_1360\1203820933.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\13156\AppData\Local\Temp\ipykernel_1360\1203820933.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Related validation work

The peer-label analysis above does not check the LLM's facet extraction. Riya's facet-fill rubric and Yiming's `v0/verification/` compare individual facet claims with dated source evidence; Yiming reports problems in 24.5% of 208 checkable facets across 20 companies. His name-masked website-only test filled 61% of facets. Yifan and Richard's separate `verification/gold-labels` branch adds a 40-company benchmark and RAG evidence retrieval.

Kethan's `v0/peer_evaluation.py` judges accepted peers against saved website passages. Its original 663-pair run used a 0–3 rubric and the earlier size split. The 0–2 evaluation below reuses its cited-evidence process on the current 36 reviewed pairs. Neither run changes the 160-target SIC2 holdout valuation.

References: `v0/verification/README.md`, `v0/PEER_EVALUATION.md`, and the 27 October team standups in the Rational AI Google Chat.

In [4]:
checks = pd.DataFrame([
    {"component": "facet claims", "owner": "Riya / Yiming", "unit": "company × facet",
     "local code": "v0/verification/", "used here": "context; separate gold"},
    {"component": "RAG evidence", "owner": "Yifan / Richard", "unit": "company × facet",
     "local code": "verification/gold-labels branch", "used here": "not merged"},
    {"component": "LLM peer judge", "owner": "Kethan", "unit": "accepted target × peer",
     "local code": "v0/peer_evaluation.py", "used here": "36-pair 0–2 diagnostic"},
    {"component": "three-reviewer gold", "owner": "Riya / Barbara / Matthew",
     "unit": "target × peer", "local code": "v0/reviewer_gold_eval.py",
     "used here": "facet comparison"},
])
display(checks)

,component,owner,unit,local code,used here
0,facet claims,Riya / Yiming,company × facet,v0/verification/,context; separate gold
1,RAG evidence,Yifan / Richard,company × facet,verification/gold-labels branch,not merged
2,LLM peer judge,Kethan,accepted target × peer,v0/peer_evaluation.py,older size split
3,three-reviewer gold,Riya / Barbara / Matthew,target × peer,v0/reviewer_gold_eval.py,facet comparison


### Source-cited peer judge: 0–2 comparison

The shared rubric defines **2** as the same product *and* customers, **1** as overlap in product *or* customers but not both, and **0** as no real overlap. `v0/judge_gold_eval.py` applies these definitions directly to the 36 PCTY/WEN pairs. It reuses Kethan's saved-page evidence and citation checks. Reviewer scores and financial data are excluded from the judge input; the three human scores remain separate in the results.

The judge scored all 36 pairs. Exact agreement is 15/36 with Riya, 26/36 with Matthew, and 32/36 with Barbara. At least two reviewers agree on 32 pairs; the judge agrees with that majority on 27/32. Four pairs have no majority, and five have a judge–majority disagreement. This is a two-target development diagnostic, not a holdout test.

To regenerate the labels from the repository root: `python v0/judge_gold_eval.py --execute --workers 3`. The following cell reads the saved results, displays every label, and plots agreement by reviewer.


In [ ]:
judge_dir = OUT / "golden_set/judge_comparison_0_2_exact/tables"
judge_summary = json.loads((judge_dir / "summary.json").read_text())
judge_agreement = pd.read_csv(judge_dir / "agreement.csv")
judge_labels = pd.read_csv(judge_dir / "labels_side_by_side.csv")
judge_review = pd.read_csv(judge_dir / "review_queue.csv")

assert judge_summary["pairs"] == 36 and judge_summary["graded"] == 36
assert len(judge_labels) == 36

display(judge_agreement[["reference", "matching_pairs", "graded_pairs",
                         "exact_agreement", "mean_absolute_grade_error"]].round(3))
display(judge_labels[["pair", "score_riya", "score_matthew", "score_barbara",
                      "human_average", "human_majority", "judge_score_0_2",
                      "judge_reason"]])
display(judge_review[["pair", "human_majority", "judge_score_0_2",
                      "judge_reason"]])

fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.bar(judge_agreement["reference"], judge_agreement["exact_agreement"])
ax.bar_label(bars, fmt="%.1f%%", labels=[
    f"{100 * value:.1f}%" for value in judge_agreement["exact_agreement"]
])
ax.set_ylim(0, 1)
ax.set_ylabel("Exact agreement")
ax.set_title("Source-cited judge versus human peer labels")
plt.tight_layout()
plt.show()


## 3. Business-structure evidence

The source-linked check distinguishes concurrent operating activities from product variants and from formerly owned businesses. It keeps all activities under the original company ID. The ten-case source audit is a targeted regression test, not an estimate of population accuracy. A separate paired trial found that inserting this context into the existing judge prompt worsened valuation error, so the default matcher remains unchanged.

The source groups also permit a narrow post-selection diagnostic: flag peers among the 40 audited companies that have multiple unrelated businesses, then compare estimates after excluding them. For example, AIR's direct LLM list includes BUKS; Butler reports both aerospace and gaming operations, with approximately 62% and 38% of FY2026 revenue, respectively, so its whole-company multiple is not a pure aerospace multiple ([company investor page](https://butlernational.com/investing/)). This 40-company check does not classify the full peer universe.

In [5]:
source = json.loads((SPLIT / "source_benchmark/summary.json").read_text())
peer = json.loads((SPLIT / "peer_test/summary.json").read_text())
trial = json.loads((SPLIT / "holdout_test/summary.json").read_text())
display(pd.DataFrame({
    "check": ["activity status (34 labels)", "exact business groups (9 cases)"],
    "saved": [f"{source['activity_correct_saved']}/34", f"{source['group_exact_saved']}/9"],
    "source checked": [f"{source['activity_correct_refined']}/34",
                       f"{source['group_exact_refined']}/9"],
}))
print(f"Paired development trial: {peer['control']['median_ape']:.2f}% → "
      f"{peer['selective_scope']['median_ape']:.2f}% median APE; "
      f"six-target pilot holdout: {trial['control']['median_ape']:.2f}% → "
      f"{trial['selective_scope']['median_ape']:.2f}%.")
scope_result = HOLDOUT / "evaluation/source_scope_sensitivity/sensitivity.csv"
if scope_result.exists():
    print("Known mixed-business peer exclusion (sensitivity only):")
    display(pd.read_csv(scope_result).round(2))

,check,saved,source checked
0,activity status (34 labels),27/34,34/34
1,exact business groups (9 cases),7/9,9/9


Paired development trial: 19.26% → 39.41% median APE; six-target pilot holdout: 31.02% → 40.21%.
Known mixed-business peer exclusion (sensitivity only):


,route,source-audited mixed peers,affected targets,affected common targets,original coverage,filtered coverage,common targets,original median APE on common,filtered median APE on common,median paired change,affected median APE original,affected median APE filtered,affected median paired change,affected improved,affected worsened
0,cosine z >= 2,62,48,48,160,160,160,37.32,37.98,0.0,32.11,31.76,-0.27,27,21
1,cosine then LLM,4,4,3,110,109,109,37.50,37.55,0.0,51.22,51.35,3.57,1,2
2,LLM direct,6,6,5,119,118,118,33.42,33.42,0.0,39.29,33.85,-0.31,3,2
3,LLM then cosine,6,6,5,119,118,118,33.59,33.18,0.0,37.93,32.97,-0.23,3,2


## 4. Four peer-selection routes

1. **Cosine:** rank all training firms by the mean of available facet similarities.
2. **Cosine → LLM:** the LLM accepts or rejects cosine candidates at z ≥ 2.
3. **Direct LLM:** the LLM selects up to ten firms from the complete training pool.
4. **LLM → cosine:** cosine scores rerank and weight the direct selections.

The four routes use the same holdout targets and pool. The holdout calls batch independent targets to avoid repeating a large pool prompt; this differs from the original Haiku one-target-per-call training protocol. Missing peers are reported as coverage loss, without an unreported fallback. The reviewer-based F3 ablation is shown beside these routes as a prespecified candidate change.

In [6]:
llm_run = json.loads((HOLDOUT / "llm_run.json").read_text())
issue_path = HOLDOUT / "answer_issues.csv"
issues = pd.read_csv(issue_path) if issue_path.exists() else pd.DataFrame()
llm_run.setdefault("answer_issues", len(issues))
llm_run.setdefault("inference_protocol", "one target per call")
display(pd.DataFrame([llm_run])[["model", "inference_protocol", "targets_done", "holdout_total",
                                 "training_pool", "judged_pairs", "accepted_judge_pairs",
                                 "direct_pairs", "answer_issues", "complete"]])
print(f"Invalid or repeated labels discarded: {len(issues)}")

,model,inference_protocol,targets_done,holdout_total,training_pool,judged_pairs,accepted_judge_pairs,direct_pairs,answer_issues,complete
0,gpt-6.1-sol,batched independent targets,160,160,373,2138,281,311,0,True


Invalid or repeated labels discarded: 0


In [7]:
universe = pd.read_csv(ROOT / "outputs-local/comps/universe" / candidate_run["universe_run"] /
                       "universe.csv", dtype=str).set_index("IQ_COMPANY_ID")
air_id = universe.index[universe["target_ciq_ticker"].eq("AIR")][0]
direct_peers = pd.read_csv(HOLDOUT / "direct.csv", dtype={"target": str, "comp": str})
judged_peers = pd.read_csv(HOLDOUT / "judge.csv", dtype={"target": str, "comp": str})
print("AIR: direct selections and their recorded reasons")
example = direct_peers[direct_peers["target"].eq(air_id)].copy()
example["peer ticker"] = example["comp"].map(universe["target_ciq_ticker"])
display(example[["rank", "peer ticker", "reason"]].head(10))
print("AIR: cosine shortlist decisions")
example = judged_peers[judged_peers["target"].eq(air_id)].copy()
example["peer ticker"] = example["comp"].map(universe["target_ciq_ticker"])
display(example[["peer ticker", "score", "reason"]].head(10))

AIR: direct selections and their recorded reasons


,rank,peer ticker,reason
0,1,ASLE,Supplies aftermarket aircraft parts and provides co...
1,2,WLFC,"Provides aircraft engine maintenance, repair and us..."
2,3,BUKS,"Provides aircraft maintenance, repair and modificat..."


AIR: cosine shortlist decisions


,peer ticker,score,reason
0,MSM,0,Distributes metalworking tools and industrial consu...
1,GOGO,0,Provides aircraft connectivity hardware and network...
2,CR,0,Primarily manufactures aerospace components and ind...
3,EML,0,Manufactures industrial hardware and metal componen...
4,DSGR,0,Distributes general industrial supplies and test eq...
5,EACO,0,Distributes electronic components to manufacturers ...
6,ESE,0,"Manufactures utility, filtration and aerospace equi..."
7,ISSC,0,Designs and manufactures proprietary avionics rathe...
8,AME,0,Manufactures diversified instruments and electromec...
9,UFPT,0,"Manufactures custom foam, plastic and composite com..."


## 5. Holdout valuation

The team’s `valuation.estimate` calculates a peer mean, similarity-weighted mean, or median EV/EBITDA. The no-matching control uses the median of the 373 training companies. The same 160 holdout target multiples are the answer key for every route. Median absolute percentage error (APE) is shown with the number of targets that received an estimate; those counts matter when routes abstain.

In [8]:
evaluation = pd.read_csv(HOLDOUT / "evaluation/summary.csv")
columns = ["comp rule", "targets", "median abs % error", "% within 25%",
           "% closer than no matching"]
display(evaluation[columns].round(2))
print("Prespecified F3 ablation on identical targets:")
display(pd.read_csv(HOLDOUT / "evaluation/paired_f3.csv").round(2))
print("Four routes on their common target set:")
common = HOLDOUT / "evaluation/route_common_support.csv"
if common.exists():
    display(pd.read_csv(common).round(2))
    print("Direct LLM versus LLM followed by cosine on identical targets:")
    display(pd.read_csv(HOLDOUT / "evaluation/reverse_hybrid_paired.csv").round(2))
else:
    print("LLM holdout routes are still running.")

view = evaluation[evaluation["comp rule"].isin([
    "no matching, training-pool median", "cosine z >= 2, weighted mean",
    "cosine minus F3 z >= 2, weighted mean", "cosine then LLM, weighted mean",
    "LLM direct, mean", "LLM then cosine, weighted mean"])]
labels = {"no matching, training-pool median": "No matching",
          "cosine z >= 2, weighted mean": "Cosine",
          "cosine minus F3 z >= 2, weighted mean": "Cosine without F3",
          "cosine then LLM, weighted mean": "Cosine → LLM",
          "LLM direct, mean": "Direct LLM",
          "LLM then cosine, weighted mean": "LLM → cosine"}
colors = ["#8a9299", "#397aa6", "#c48635", "#397aa6", "#397aa6", "#397aa6"]
fig, ax = plt.subplots(figsize=(8.5, 4))
bars = ax.barh(view["comp rule"].map(labels), view["median abs % error"], color=colors)
for bar, n in zip(bars, view["targets"]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f"{bar.get_width():.1f}%  ({n}/160)", va="center", fontsize=9)
ax.set_xlabel("Median absolute percentage error (%)")
ax.invert_yaxis()
ax.set_xlim(0, max(view["median abs % error"]) * 1.42)
fig.tight_layout()
fig.savefig(HOLDOUT / "evaluation/holdout_methods.png", dpi=180)
plt.show()

,comp rule,targets,median abs % error,% within 25%,% closer than no matching
0,"no matching, training-pool median",160,39.92,34.38,NaN
1,"cosine top 10, weighted mean",160,36.44,35.62,58.13
2,"cosine top 10, median",160,34.70,36.25,59.38
3,"cosine z >= 2, weighted mean",160,37.32,36.25,50.62
4,"cosine z >= 2, median",160,36.08,36.88,54.37
5,"cosine z >= 2.5, weighted mean",158,31.32,42.41,56.96
6,"cosine z >= 2.5, median",158,29.33,42.41,55.06
7,"cosine minus F3 top 10, weighted mean",160,35.18,40.62,58.13
8,"cosine minus F3 top 10, median",160,35.39,36.25,56.88
9,"cosine minus F3 z >= 2, weighted mean",160,33.29,38.12,57.50


Prespecified F3 ablation on identical targets:


,selection,estimator,common targets,old median APE,minus F3 median APE,median paired change,paired change 95% low,paired change 95% high,targets improved,targets worsened
0,top 10,weighted mean,160,36.44,35.18,-0.07,-0.27,0.08,87,73
1,top 10,median,160,34.70,35.39,0.00,0.00,0.00,46,42
2,z >= 2,weighted mean,160,37.32,33.29,-0.15,-2.04,0.08,89,71
3,z >= 2,median,160,36.08,31.75,0.00,-0.61,0.00,72,59
4,z >= 2.5,weighted mean,158,31.32,31.75,0.04,-0.04,0.32,69,88
5,z >= 2.5,median,158,29.33,29.96,0.00,0.00,0.00,43,54


Four routes on their common target set:


,method,common targets,median APE on common targets,median paired change vs cosine,paired change 95% low,paired change 95% high,targets improved vs cosine
0,"cosine z >= 2, weighted mean",107,36.83,0.00,0.00,0.00,0
1,"cosine then LLM, weighted mean",107,37.50,1.12,-3.07,4.75,49
2,"LLM direct, mean",107,37.79,1.55,-4.14,4.68,51
3,"LLM then cosine, weighted mean",107,37.93,1.12,-4.74,4.72,49


Direct LLM versus LLM followed by cosine on identical targets:


,common targets,direct median APE,LLM then cosine median APE,median paired change,paired change 95% low,paired change 95% high,targets improved,targets worsened
0,119,33.78,33.78,0.0,0.0,0.0,40,42


C:\Users\13156\AppData\Local\Temp\ipykernel_1360\2374802254.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Optional route: source-cited grades in peer selection

This experiment keeps the SIC2 split, 373 training firms, cosine z ≥ 2 candidate pool, and valuation function fixed. For each holdout target, the judge scores every candidate against saved website passages under the shared 0–2 rubric. It sees no similarity scores, peer ranks, reviewer labels, or financial fields. A single target-level call returns independent, source-cited grades for its candidates; each company contributes at most 900 words of saved passages. This batching protocol differs from the individual 1,500-word calls in the diagnostic above.

Two prespecified selection rules are tested: **2 only**, and **2 followed by 1**. Both exclude 0 and insufficient evidence, order peers by grade and then cosine similarity, and retain at most ten. The grade is ordinal; only cosine supplies valuation weights. Zero-peer targets have no peer estimate and are counted in coverage. The existing four routes and no-matching control remain unchanged.

The batched judge scored all 36 PCTY/WEN development pairs: 26/32 agreement with defined reviewer majorities and 11/11 with unanimous labels. Its grade matches the individual-call diagnostic on 31/36 pairs. Peer-ranking metrics below retain each reviewer's labels separately. Holdout errors are reported both on each method's covered targets and as paired changes on identical targets.

Across 2,138 holdout candidate pairs, the batched judge assigned 327 scores of 2, 831 scores of 1, 824 scores of 0, and 156 insufficient-evidence responses. The 2-only rule estimates 116/160 targets (72.5% coverage; 37.18% median APE on those 116). The 2-then-1 rule estimates 155/160 (96.9% coverage; 37.32% median APE on those 155). The original binary judge covers 110/160; cosine covers all 160. On the 155 targets shared with cosine, the 2-then-1 rule's median paired APE change is 0.00 percentage points (bootstrap 95% interval −1.86 to +1.83); 75 targets improve and 76 worsen. It expands coverage relative to the binary judge without an established error reduction. Four of its five no-peer targets lack usable saved target-site passages; the fifth has only 0-rated candidates. Rebuild these tables without model calls using `python v0/graded_peer_selection.py --mode holdout --evaluate-saved`.


In [ ]:
graded_dir = HOLDOUT / "graded_peer_selection"
graded_eval = pd.read_csv(graded_dir / "evaluation.csv")
graded_paired = pd.read_csv(graded_dir / "paired_comparison.csv")
graded_gold = pd.read_csv(OUT / "golden_set/graded_batch_check/gold_selection_metrics.csv")
graded_coverage = pd.read_csv(graded_dir / "coverage_details.csv")

display(graded_eval[["method", "targets", "coverage", "median_APE",
                     "median_comps", "common_targets_with_cosine",
                     "median_paired_APE_change_vs_cosine"]].round(2))
display(graded_paired[["new_method", "comparison_method", "common_targets",
                      "median_paired_change", "paired_change_95_low",
                      "paired_change_95_high", "targets_improved",
                      "targets_worsened"]].round(2))
gold_means = graded_gold.loc[graded_gold.target.eq("mean")]
display(gold_means.pivot(index="route", columns="reviewer", values="NDCG@10").round(3))
display(graded_coverage.loc[graded_coverage.selected_2_then_1.eq(0),
                            ["ticker", "target_website_status", "score_0_candidates",
                             "insufficient_evidence_candidates"]])

fig, ax = plt.subplots(figsize=(9, 3))
bars = ax.bar(graded_eval["method"], 100 * graded_eval["coverage"])
ax.bar_label(bars, fmt="%.0f%%")
ax.set_ylim(0, 110)
ax.set_ylabel("Targets with a peer estimate (%)")
ax.set_title("Holdout coverage by selection route")
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()


## 6. Interpretation and next decision

The two-target reviewer set favors removing F3 (`primary_services`): majority-label pairwise accuracy rises from 0.851 to 0.921, and all three individual reviewers also improve. On the 160-target holdout, cosine at z ≥ 2 improves from 37.32% to 33.29% median APE after removing F3, but the bootstrap interval for the paired median change includes zero (−2.04 to +0.08 percentage points). The effect reverses under the z ≥ 2.5 rule on its common 158 targets. F3 removal remains a candidate, not a new default.

On the 107 targets covered by all four routes, cosine has 36.83% median APE; cosine → LLM has 37.50%, direct LLM 37.79%, and LLM → cosine 37.93%. Their paired intervals against cosine include zero. Direct LLM and LLM → cosine both cover 119/160 and have 33.78% median APE on those 119; reranking improves 40 targets and worsens 42. The fourth route has no demonstrated holdout advantage.

The source-backed grouping corrects the targeted activity and business partitions, but adding scope text to the judge worsened the paired pilot results. Excluding source-confirmed mixed-business peers changes cosine median APE from 37.32% to 37.98% over all 160 targets; among 48 affected targets, 27 improve and 21 worsen. Only 4–6 targets are affected in each LLM route, and filtering loses one estimate per route. Keep the business groups as auditable metadata while extending the peer universe beyond 560 and testing website-observable private-company inputs. The public holdout does not establish performance on private transactions.

The individual-call, source-cited judge agrees with the defined human majority on 27/32 PCTY/WEN pairs; the batched protocol agrees on 26/32. The optional 2-then-1 selection route improves holdout estimate coverage to 155/160, but its paired valuation-error interval versus cosine includes zero. It remains an experiment rather than a replacement for the four original routes. Source access, rather than the ranking rule, accounts for four of its five unestimated targets.

**Source files:** `v0/reviewer_gold_eval.py`, `v0/judge_gold_eval.py`, `v0/graded_peer_selection.py`, `v0/facet_eval.py`, `v0/business_split_refinement.py`, `v0/holdout_split_candidates.py`, `v0/holdout_batched_routes.py`, `v0/holdout_evaluation.py`, and `v0/source_scope_sensitivity.py`. Team inputs are the saved v0 facet/candidate/financial tables and the three-reviewer golden-set Sheet snapshot. `v0/README.md` records the original route definitions.